# 01 - Data Exploration

This notebook explores the existing datasets in your project.

**Datasets Available:**
- Severity Classification Data (560,486 samples, 972 features)
- ETA Prediction Data (1,447,755 trips)
- Bed Availability Data (57,649 records)
- Hospital Registry (10 hospitals)

## Cell 1: Setup - Mount Google Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set project directory
import os
PROJECT_DIR = '/content/drive/MyDrive/smart-ambulance-ml'
os.makedirs(f"{PROJECT_DIR}/results", exist_ok=True)

# Install required packages
!pip install -q numpy pandas matplotlib seaborn

print("✓ Setup complete!")

## Cell 2: Load and Explore Severity Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("="*60)
print("SEVERITY DATA EXPLORATION")
print("="*60)

# Load processed severity data
severity_df = pd.read_csv(f'{PROJECT_DIR}/datasets/processed/severity_processed.csv')

print(f"\nShape: {severity_df.shape}")
print(f"Columns: {len(severity_df.columns)}")
print(f"\nColumn names (first 20):")
print(list(severity_df.columns[:20]))

# Check for target column
if 'esi' in severity_df.columns:
    print(f"\n✓ Target column 'esi' found")
    print(f"ESI distribution:")
    print(severity_df['esi'].value_counts().sort_index())
else:
    print("\n⚠ Target column 'esi' not found. Available columns:")
    print([c for c in severity_df.columns if 'esi' in c.lower() or 'severity' in c.lower()][:10])

# Basic statistics
print(f"\nData types summary:")
print(severity_df.dtypes.value_counts())
print(f"\nMissing values: {severity_df.isnull().sum().sum()}")
print(f"Duplicate rows: {severity_df.duplicated().sum()}")

## Cell 3: Load and Explore ETA Data

In [ ]:
print("\n="*60)
print("ETA DATA EXPLORATION")
print("="*60)

# Load ETA data
eta_df = pd.read_csv(f'{PROJECT_DIR}/datasets/processed/eta_processed.csv')

print(f"\nShape: {eta_df.shape}")
print(f"\nColumns:")
print(list(eta_df.columns))

# Check for target column
target_col = None
for col in ['duration_min', 'trip_duration_minutes', 'duration', 'eta', 'trip_time']:
    if col in eta_df.columns:
        target_col = col
        break

if target_col:
    print(f"\n✓ Target column '{target_col}' found")
    print(f"Target stats:")
    print(eta_df[target_col].describe())
else:
    print("\n⚠ Target column not found. Available columns:")
    print(list(eta_df.columns))

# Feature summary
print(f"\nNumeric features:")
numeric_cols = eta_df.select_dtypes(include=[np.number]).columns
print(f"  Count: {len(numeric_cols)}")
print(f"  Names: {list(numeric_cols)}")

## Cell 4: Load and Explore Bed Availability Data

In [ ]:
print("\n="*60)
print("BED AVAILABILITY DATA EXPLORATION")
print("="*60)

# Load bed data
bed_df = pd.read_csv(f'{PROJECT_DIR}/datasets/processed/bed_availability_processed.csv')

print(f"\nShape: {bed_df.shape}")
print(f"\nColumns:")
print(list(bed_df.columns))

# Check for target column
if 'available_beds' in bed_df.columns:
    print(f"\n✓ Target column 'available_beds' found")
    print(f"Target stats:")
    print(bed_df['available_beds'].describe())
else:
    print("\n⚠ Target column 'available_beds' not found")

print(f"\nData sample:")
print(bed_df.head(3))

## Cell 5: Load Hospital Registry

In [ ]:
print("\n="*60)
print("HOSPITAL REGISTRY")
print("="*60)

# Load hospitals
hospitals_df = pd.read_csv(f'{PROJECT_DIR}/datasets/synthetic/hospitals.csv')

print(f"\nShape: {hospitals_df.shape}")
print(f"\nColumns:")
print(list(hospitals_df.columns))

print(f"\nHospital Summary:")
print(hospitals_df[['hospital_id', 'name', 'total_icu_beds', 'total_emergency_beds', 'total_general_beds']].to_string())

print(f"\nTotal beds by hospital:")
hospitals_df['total_beds'] = hospitals_df['total_icu_beds'] + hospitals_df['total_emergency_beds'] + hospitals_df['total_general_beds']
print(hospitals_df[['name', 'total_beds']].to_string())

## Cell 6: Visualize Data Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Severity distribution
if 'esi' in severity_df.columns:
    severity_counts = severity_df['esi'].value_counts().sort_index()
    axes[0, 0].bar(severity_counts.index, severity_counts.values, color='coral')
    axes[0, 0].set_xlabel('ESI Level')
    axes[0, 0].set_ylabel('Count')
    axes[0, 0].set_title('Severity Distribution')
    for i, v in enumerate(severity_counts.values):
        axes[0, 0].text(severity_counts.index[i], v + 1000, str(v), ha='center')

# ETA distribution
if target_col:
    axes[0, 1].hist(eta_df[target_col], bins=50, color='skyblue', edgecolor='black')
    axes[0, 1].set_xlabel('Duration (minutes)')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].set_title('ETA Distribution')
    axes[0, 1].axvline(eta_df[target_col].mean(), color='red', linestyle='--', label=f"Mean: {eta_df[target_col].mean():.1f} min")
    axes[0, 1].legend()

# Bed availability distribution
if 'available_beds' in bed_df.columns:
    axes[1, 0].hist(bed_df['available_beds'], bins=30, color='seagreen', edgecolor='black')
    axes[1, 0].set_xlabel('Available Beds')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('Bed Availability Distribution')

# Hospital bed comparison
axes[1, 1].barh(hospitals_df['name'], hospitals_df['total_beds'], color='steelblue')
axes[1, 1].set_xlabel('Total Beds')
axes[1, 1].set_title('Hospital Capacity')
axes[1, 1].tick_params(axis='y', labelsize=8)

plt.suptitle('Dataset Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/results/data_exploration_overview.png', dpi=150)
plt.show()
print(f"\n✓ Plot saved to {PROJECT_DIR}/results/data_exploration_overview.png")

## Cell 7: Data Quality Summary

In [ ]:
print("="*60)
print("DATA QUALITY SUMMARY")
print("="*60)

summary = {
    'Dataset': ['Severity', 'ETA', 'Bed Availability', 'Hospitals'],
    'Rows': [severity_df.shape[0], eta_df.shape[0], bed_df.shape[0], hospitals_df.shape[0]],
    'Columns': [severity_df.shape[1], eta_df.shape[1], bed_df.shape[1], hospitals_df.shape[1]],
    'Target': ['esi', target_col or 'N/A', 'available_beds', 'N/A']
}

summary_df = pd.DataFrame(summary)
print("\nDataset Summary:")
print(summary_df.to_string(index=False))

print("\n" + "="*60)
print("DATA EXPLORATION COMPLETE")
print("="*60)
print("\nNext steps:")
print("  1. Skip Notebook 02 (data is already preprocessed)")
print("  2. Go directly to Notebook 03 for model training")